# Lonboard: Interactive Maps of Millions of Features in Jupyter

![Lonboard: US mobile download speeds](thumbnail.png)

> **Running on I-GUIDE?** Before running anything (including the `pip install` cell below), switch this notebook to the **`geoai`** conda environment: *Kernel → Change Kernel… →* select the `geoai` kernel.

Big thanks to the [Lonboard](https://github.com/developmentseed/lonboard) maintainers at Development Seed! The data and much of the code here are adapted from Lonboard's own examples: [`internet-speeds`](https://developmentseed.org/lonboard/latest/examples/internet-speeds/), [`data-filter-extension`](https://developmentseed.org/lonboard/latest/examples/data-filter-extension/), [`column-layer`](https://developmentseed.org/lonboard/latest/examples/column-layer/) (3D view) and [`duckdb`](https://developmentseed.org/lonboard/latest/examples/duckdb/).

## Why another mapping library?

Most Python web-map libraries (`folium`, `ipyleaflet`, `geopandas.explore()`) work the same way: they convert your data to **GeoJSON**, a text format, and the browser parses it and draws each feature one by one. That works fine up to roughly 10k–100k features, and then the notebook freezes.

**Lonboard** takes a different route:

| | **GeoJSON-based** (folium, ipyleaflet, …) | **Lonboard** |
|---|---|---|
| Data format sent to the browser | GeoJSON text (every coordinate written out as digits) | **GeoArrow** in compressed **Parquet** (binary, columnar) |
| Parsing in the browser | `JSON.parse` of a giant string | Almost none: Arrow buffers go straight to the GPU |
| Rendering | DOM / Canvas, feature by feature | **GPU** via [deck.gl](https://deck.gl) |
| Practical limit | ~10k–100k features | **Millions** of features |

It is the same columnar Arrow idea that makes Polars fast, applied to visualization.

**Plan (~15 min):** data (1 min) → one-liner (1) → why it's fast (2) → styling (2) → GPU filtering (2) → 3D polygons (2) → DuckDB (2) → selection & export (2) → wrap-up (1)

## Setup

Make sure you are in the **`geoai`** environment (see the note at the top). Then install the packages and **restart the kernel** (Kernel → Restart) before running the rest of the notebook.

- `-U` upgrades packages that are already installed. Shared hubs (e.g., I-GUIDE) often come with an old `pyarrow`, and a plain `pip install` would leave it alone.
- `lonboard[geopandas]` pulls in the versions of `pyarrow`, `pandas` and `shapely` that Lonboard needs.
- The restart is needed because a running kernel keeps using the old versions it already loaded.

In [ ]:
# Install / upgrade packages (only needed once), then restart the kernel
!pip install -U "lonboard[geopandas]" "pyarrow>=16" duckdb palettable matplotlib

In [ ]:
from io import BytesIO
from pathlib import Path
import time

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shapely
import pyarrow
import lonboard

# Lonboard needs pyarrow >= 16 to read GeoDataFrames. If this fails, re-run the
# install cell above and restart the kernel.
print(f"lonboard {lonboard.__version__}, pyarrow {pyarrow.__version__}, geopandas {gpd.__version__}")
if int(pyarrow.__version__.split(".")[0]) < 16:
    raise RuntimeError(
        f"pyarrow {pyarrow.__version__} is too old for Lonboard (need >= 16). "
        "Run the install cell above, then restart the kernel."
    )

## Data: Ookla global mobile network speeds

This dataset comes from Lonboard's flagship example. It is from [Ookla's Speedtest app](https://registry.opendata.aws/speedtest-global-performance/) via the AWS Open Data Registry:

> Global fixed broadband and mobile (cellular) network performance, allocated to zoom level 16 web mercator tiles (approximately 610.8 meters by 610.8 meters at the equator). Download speed, upload speed, and latency are collected via the Speedtest by Ookla applications for Android and iOS and averaged for each tile.

We use **mobile, Q1 2019**: about **3.2 million tiles worldwide**. Each tile is stored as a WKT polygon string. We parse it once and save two local GeoParquet files:

- `ookla-points.parquet`: every tile's **centroid** (most of the notebook uses points)
- `ookla-chicago-tiles.parquet`: the actual **tile polygons** for the Chicago area (for the 3D section)

The first run downloads about 50 MB of columns straight from S3 and takes around 1–2 minutes. After that, everything loads from the local cache.

In [ ]:
url = "https://ookla-open-data.s3.us-west-2.amazonaws.com/parquet/performance/type=mobile/year=2019/quarter=1/2019-01-01_performance_mobile_tiles.parquet"

Path("data").mkdir(exist_ok=True)
points_path = Path("data/ookla-points.parquet")
chicago_path = Path("data/ookla-chicago-tiles.parquet")
CHICAGO = (-88.3, 41.6, -87.5, 42.1)  # (minx, miny, maxx, maxy)

if not (points_path.exists() and chicago_path.exists()):
    columns = ["avg_d_kbps", "avg_u_kbps", "avg_lat_ms", "tests", "devices", "tile"]
    df = pd.read_parquet(url, columns=columns)

    # WKT strings -> polygons -> centroids
    tiles = shapely.from_wkt(df.pop("tile"))
    centroids = shapely.centroid(tiles)
    gpd.GeoDataFrame(df, geometry=centroids, crs="EPSG:4326").to_parquet(points_path)

    # Keep the full polygons only for the Chicago area
    x, y = shapely.get_x(centroids), shapely.get_y(centroids)
    in_chicago = (x >= CHICAGO[0]) & (x <= CHICAGO[2]) & (y >= CHICAGO[1]) & (y <= CHICAGO[3])
    gpd.GeoDataFrame(df[in_chicago], geometry=tiles[in_chicago], crs="EPSG:4326").to_parquet(chicago_path)
    del df, tiles, centroids

world = gpd.read_parquet(points_path)
print(f"{len(world):,} tiles worldwide")
world.head()

We will focus on the **contiguous US**. To clip the points we use US state boundaries from [Natural Earth](https://www.naturalearthdata.com/), the same source as Lonboard's `north-america-roads` example. We also convert speeds from kbps to **Mbps**, which is easier to read.

In [ ]:
states = gpd.read_file(
    "https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_1_states_provinces.zip"
)
states = states[(states["iso_a2"] == "US") & ~states["name"].isin(["Alaska", "Hawaii"])]
states = states[["name", "geometry"]]

# Spatial join: keep only tiles that fall inside a contiguous-US state
us = (
    world.cx[-125:-66.9, 24.4:49.4]
    .sjoin(states, predicate="within")
    .drop(columns="index_right")
    .reset_index(drop=True)
)
us["download_mbps"] = us["avg_d_kbps"] / 1000
us["upload_mbps"] = us["avg_u_kbps"] / 1000

print(f"{len(us):,} tiles in the contiguous US")
us[["name", "download_mbps", "upload_mbps", "avg_lat_ms", "tests", "geometry"]].head()

---
## 1. The one-liner: `viz`

`viz` inspects the geometry type and picks a sensible layer for you. Here that is almost **half a million points**, rendered on the GPU.

**Try:** pan and zoom, then **click a point** to see its attributes in the side panel.

In [ ]:
from lonboard import viz

viz(us)

## 2. Why is it fast? GeoJSON vs. GeoArrow

Let's measure what each approach would send to the browser for the same ~460k points:

- **GeoJSON**: what `folium` / `ipyleaflet` / `.explore()` embed in the notebook
- **GeoParquet**: what Lonboard sends, a compressed binary columnar format

In [ ]:
start = time.time()
geojson_bytes = len(us.to_json().encode())
geojson_time = time.time() - start

start = time.time()
buffer = BytesIO()
us.to_parquet(buffer, compression="zstd")
parquet_bytes = buffer.getbuffer().nbytes
parquet_time = time.time() - start

print(f"GeoJSON:    {geojson_bytes / 1e6:7.1f} MB  (serialized in {geojson_time:.1f}s)")
print(f"GeoParquet: {parquet_bytes / 1e6:7.1f} MB  (serialized in {parquet_time:.1f}s)")
print(f"-> {geojson_bytes / parquet_bytes:.0f}x smaller, {geojson_time / parquet_time:.0f}x faster to serialize")

And that is only the transfer. On top of this, the browser has to `JSON.parse` the GeoJSON string and build JavaScript objects for every feature. The Arrow buffers, by contrast, are copied almost directly into GPU memory.

> Feel free to try `us.explore()` (folium) on this data, but expect your browser tab to freeze. 🙂

## 3. Explicit layers + data-driven styling

For more control, build the layer yourself. Lonboard has many [layer types](https://developmentseed.org/lonboard/latest/api/layers/scatterplot-layer/), including `ScatterplotLayer`, `PathLayer`, `PolygonLayer`, `HeatmapLayer`, `H3HexagonLayer`, `ArcLayer` and `TripsLayer`. Any property starting with `get_*` is an **accessor**: it takes either one constant value or **one value per row** (a NumPy array).

We color by download speed. `apply_continuous_cmap` expects values normalized to **0–1**. Anything outside that range is clamped to the end colors.

In [ ]:
from lonboard import Map, ScatterplotLayer
from lonboard.basemap import CartoStyle, MaplibreBasemap
from lonboard.colormap import apply_continuous_cmap

# Normalize download speed: 0 Mbps -> 0, 50+ Mbps -> 1
normalized = np.clip(us["download_mbps"] / 50, 0, 1)
cmap = plt.colormaps["viridis"]

layer = ScatterplotLayer.from_geopandas(
    us,
    get_fill_color=apply_continuous_cmap(normalized, cmap, alpha=0.8),
    get_radius=300,
    radius_units="meters",
    radius_min_pixels=0.5,
)

m = Map(layer, basemap=MaplibreBasemap(style=CartoStyle.DarkMatter), height=500)
m

### Layers are *live* widgets

The map above is a Jupyter widget, and every layer property is a synced [traitlet](https://traitlets.readthedocs.io/). If you assign a new value in Python, the map **above** updates in place without re-rendering the cell. Here we scale the radius by speed and switch to a diverging colormap from [`palettable`](https://jiffyclub.github.io/palettable/):

In [ ]:
from palettable.colorbrewer.diverging import BrBG_10

layer.get_radius = 100 + normalized * 600
layer.get_fill_color = apply_continuous_cmap(normalized, BrBG_10, alpha=0.8)

## 4. GPU filtering with sliders

The `DataFilterExtension` filters rows **on the GPU**. Python does no recomputation and no new data is transferred; the browser simply hides points outside the range.

We filter on two attributes at once, **download speed** and **latency**, and link the sliders to the layer with `jsdlink`. That link runs entirely in the browser (no round-trip to Python), so the sliders stay smooth even with half a million points.

In [ ]:
from ipywidgets import FloatRangeSlider, jsdlink
from lonboard.controls import MultiRangeSlider
from lonboard.layer_extension import DataFilterExtension

filter_values = np.column_stack([us["download_mbps"], us["avg_lat_ms"]])
initial_range = [[0, 200], [0, 500]]

filter_layer = ScatterplotLayer.from_geopandas(
    us,
    extensions=[DataFilterExtension(filter_size=2)],
    get_filter_value=filter_values,
    filter_range=initial_range,
    get_fill_color=apply_continuous_cmap(normalized, cmap, alpha=0.8),
    get_radius=300,
    radius_units="meters",
    radius_min_pixels=0.5,
)

download_slider = FloatRangeSlider(value=initial_range[0], min=0, max=200, step=1, description="Down (Mbps)")
latency_slider = FloatRangeSlider(value=initial_range[1], min=0, max=500, step=1, description="Latency (ms)")
multi_slider = MultiRangeSlider([download_slider, latency_slider])

# Link the sliders to the layer (runs in the browser, not Python)
jsdlink((multi_slider, "value"), (filter_layer, "filter_range"))

filter_map = Map(filter_layer, basemap=MaplibreBasemap(style=CartoStyle.DarkMatter), height=500)
display(multi_slider, filter_map)

**Try:** drag *Down* to `0–5` Mbps to find the slowest tiles. Then set *Latency* to `100+` ms. Where are the bad connections concentrated?

## 5. Real tiles, in 3D

So far we drew each tile as a point. But the data is really **~600 m square tiles**, so let's draw the actual polygons for the Chicago area. With `extruded=True`, each polygon becomes a 3D column:

- **height** = number of speed tests (a rough proxy for how many people use the network there), on a **log scale** because a few tiles have thousands of tests while the median has 2
- **color** = download speed

**Try:** hold **Ctrl** (or right-click) and drag to rotate and tilt the view.

In [ ]:
from lonboard import PolygonLayer

chicago = gpd.read_parquet(chicago_path)
chicago_speed = np.clip(chicago["avg_d_kbps"] / 1000 / 50, 0, 1)
print(f"{len(chicago):,} tile polygons in the Chicago area")

tiles_3d = PolygonLayer.from_geopandas(
    chicago,
    extruded=True,
    get_elevation=np.log1p(chicago["tests"]),  # log scale: tests range from 1 to 7,000+
    elevation_scale=150,
    get_fill_color=apply_continuous_cmap(chicago_speed, cmap, alpha=0.9),
    get_line_color=[0, 0, 0, 0],
)

Map(
    tiles_3d,
    basemap=MaplibreBasemap(style=CartoStyle.DarkMatter),
    view_state={"longitude": -87.75, "latitude": 41.82, "zoom": 9.5, "pitch": 50, "bearing": -20},
    height=500,
)

Downtown (the Loop) stands out as a cluster of tall tiles (~25x more tests than a typical tile), yet its speeds are about the same as the rest of the area. Heavy use does not mean better service. **Try:** find the tallest column in the northwest suburbs and click it. That single tile logged over 7,000 tests in one quarter, an outlier you would likely miss in a summary table.

## 6. Straight from SQL: DuckDB → Lonboard

Lonboard can also read directly from [DuckDB](https://duckdb.org/) query results, **without GeoPandas in between**. DuckDB scans our 3.2M-row GeoParquet file in well under a second, and the Arrow result goes straight to the map.

Here we ask a **global** question: *where in the world is mobile internet slowest (< 2 Mbps)?* We draw the answer as a `HeatmapLayer` weighted by the number of tests.

In [ ]:
import duckdb
from lonboard import HeatmapLayer

con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")

# Materialize the query as a table (like Lonboard's duckdb example), so the
# layer and the weights below read the exact same rows in the same order
con.sql(f'''
    CREATE OR REPLACE TABLE slow AS
    SELECT
        avg_d_kbps / 1000 AS download_mbps,
        tests,
        geometry::GEOMETRY AS geometry  -- drop the CRS tag (Lonboard expects plain GEOMETRY)
    FROM '{points_path}'
    WHERE avg_d_kbps < 2000
''')
slow = con.table("slow")
print(f"{slow.count('*').fetchone()[0]:,} slow tiles worldwide")

heatmap = HeatmapLayer.from_duckdb(slow, con, crs="EPSG:4326")
heatmap.get_weight = slow.fetchnumpy()["tests"].astype(np.float32)

Map(heatmap, basemap=MaplibreBasemap(style=CartoStyle.DarkMatter), height=500)

> Any SQL works here: joins, spatial predicates (`ST_Within`, `ST_DWithin`), aggregations, or reading remote Parquet/GeoParquet over HTTP/S3. Lonboard only needs a result that has a geometry column.

## 7. Getting data *back* into Python: bounding-box selection

Interactivity runs **both ways**. Use the **bounding-box selection button** (top-right of the map) to draw a box around a region of interest. The coordinates are synced back to Python as `m.selected_bounds`, so you can analyze exactly the area you found while exploring.

👉 *Draw a box on the map from section 3 (e.g., around a city), then run the next cell.*

In [ ]:
# (minx, miny, maxx, maxy) in lon/lat, or None if nothing was drawn
if m.selected_bounds is not None:
    bbox = m.selected_bounds
else:
    bbox = CHICAGO  # fallback
    print("No selection drawn; using the Chicago bbox")

region = us.cx[bbox[0] : bbox[2], bbox[1] : bbox[3]]

summary = pd.DataFrame(
    {
        "Selected region": region[["download_mbps", "upload_mbps", "avg_lat_ms"]].median(),
        "Contiguous US": us[["download_mbps", "upload_mbps", "avg_lat_ms"]].median(),
    }
).round(1)
print(f"{len(region):,} tiles selected")
summary

## 8. Sharing: export to standalone HTML

A map can be saved as a single HTML file that anyone can open in a browser, with no Python required. The data is embedded, so keep an eye on the file size.

In [ ]:
m.to_html("us-mobile-speeds.html", title="US mobile download speeds (Ookla, Q1 2019)")
print(f"{Path('us-mobile-speeds.html').stat().st_size / 1e6:.1f} MB")

## 9. (Optional) Stress test: the whole world

All **3.2 million** tiles worldwide. This works on most recent laptops. Skip it on a weak machine or a shared hub.

In [ ]:
# viz(world)

---
## Takeaways

- **Why Lonboard:** binary GeoArrow/Parquet transfer + GPU rendering (deck.gl) → **millions of features** in a notebook, where GeoJSON-based tools stall at ~100k.
- **Quick start:** `viz(gdf)`. For control: `ScatterplotLayer` / `PolygonLayer` / `HeatmapLayer` / … `.from_geopandas(gdf)` → `Map(layer)`.
- **Styling:** every `get_*` accessor takes a constant or a per-row array. Use `apply_continuous_cmap` for colormaps.
- **Interactivity:** layers are live widgets. You can filter on the GPU with `DataFilterExtension`, and selections (`selected_bounds`) flow back to Python.
- **Ecosystem:** reads GeoPandas, DuckDB and GeoArrow directly; exports to standalone HTML.
- **Gotchas:** needs a browser with WebGL; some JupyterHubs block custom widgets; very large exports make very large HTML files.

### Links
- Lonboard docs & examples: https://developmentseed.org/lonboard/
- deck.gl layer catalog: https://deck.gl/docs/api-reference/layers
- Ookla open data: https://registry.opendata.aws/speedtest-global-performance/